In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="12_9XYzWU8FyTikLdrn7sTQofHu4Z-OSv", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_01_intro.mp3"))

# Structured Output with tool_use

**Notebook 2 of 4** | Prompt Engineering & Structured Output | Claude Certified Architect Prep

---

**Objective:** Define JSON schemas that force Claude to return structured, schema-compliant output every time. Learn to choose between auto, any, and forced tool_choice. Design robust schemas with nullable fields and enum patterns.

**Exam Task Covered:** 4.3 — tool_use with JSON schemas, tool_choice modes

**Prerequisites:** Notebook 01 (Explicit Criteria & Few-Shot Prompting)

---

In Notebook 01, we learned to write explicit criteria and few-shot examples. But even the best prompt occasionally returns free-text instead of JSON, or JSON with the wrong field names. In this notebook, we solve that problem permanently with **tool_use** — Claude's mechanism for guaranteed structured output.

By the end of this notebook, you will be able to:
1. Define extraction tools with precise JSON schemas
2. Choose between `auto`, `any`, and forced `tool_choice`
3. Design schemas with nullable fields, enum+"other" patterns, and confidence fields
4. Distinguish syntax errors (solved by schemas) from semantic errors (require validation)

In [ ]:
#@title 🎧 Listen: Concept Overview
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_concept_overview.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Syntax Vs Semantic
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_syntax_vs_semantic.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Concept Overview

### The Problem: "Please Return JSON" Is Unreliable

Consider this common pattern:

```python
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    messages=[{"role": "user", "content": "Extract invoice data. Return JSON."}]
)
data = json.loads(response.content[0].text)  # Sometimes fails!
```

Even with explicit format instructions, Claude occasionally:
- Returns a preamble before the JSON ("Here is the extracted data: {...}")
- Uses slightly different field names than expected (`total` vs `total_amount`)
- Omits fields or adds unexpected ones
- Returns valid JSON with the wrong structure (flat when you need nested)

In production, a **~8% failure rate** on JSON parsing means your pipeline crashes multiple times per day.

### The Fix: tool_use as a Structured Output Mechanism

Instead of *asking* Claude to return JSON, you **define a tool** with a JSON schema. Claude "calls" the tool, which means it *must* return data that matches the schema exactly.

```
Traditional approach:  "Return JSON with these fields..."  -> Hope for the best
tool_use approach:     Define a tool with input_schema     -> Guaranteed compliance
```

The tool does not need to do anything. It is a structural contract — a way to tell Claude "your output must look exactly like this."

### The Three tool_choice Modes

When you send tools to Claude, you also specify `tool_choice`, which controls when and how tools are used:

| Mode | Syntax | Behavior | Use Case |
|---|---|---|---|
| **auto** | `tool_choice={"type": "auto"}` | Model decides whether to use a tool | Chatbots with optional tools |
| **any** | `tool_choice={"type": "any"}` | Must call *some* tool, model picks which | Multi-schema routing |
| **forced** | `tool_choice={"type": "tool", "name": "..."}` | Must call this *specific* tool | Extraction pipelines |

For extraction, **always use forced**. There is no reason to accept a free-text response when you need structured data.

### What Schemas Guarantee (and What They Do Not)

Schemas guarantee **syntax** — the JSON will always be valid, field names will always match, types will always be correct.

Schemas do **not** guarantee **semantics** — the values might be wrong. An invoice total of $500 when the line items sum to $450 is a semantic error that passes schema validation perfectly.

| Error Type | Schema Catches It? | Example |
|---|---|---|
| Missing field | Yes | `total_amount` not present |
| Wrong type | Yes | `total_amount: "five hundred"` instead of `500.0` |
| Invalid enum | Yes | `currency: "Dollars"` when enum is `["USD","EUR","GBP"]` |
| Wrong value | **No** | `total_amount: 500` when it should be `450` |
| Math error | **No** | Line items do not sum to total |
| Hallucination | **No** | `vendor_name: "Acme"` when document says "Ajax" |

This distinction — syntax vs semantics — is critical for the exam and for production systems. We will explore it deeply in Section 4.

In [ ]:
#@title 🎧 Listen: Setup
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_setup.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## Setup

We will use a **mock client** that simulates the Anthropic `tool_use` API. This lets us run every cell without an API key while demonstrating exact production behavior. In production, replace `MockToolClient()` with `anthropic.Anthropic()`.

In [ ]:
!pip install anthropic -q

import json


# ---------------------------------------------------------------------------
# Mock Claude tool_use client for exercises
# Replace with: client = anthropic.Anthropic() for production use
# ---------------------------------------------------------------------------

class MockToolResponse:
    """Mimics anthropic.types.Message for tool_use responses."""
    def __init__(self, content, stop_reason="tool_use"):
        self.content = content
        self.stop_reason = stop_reason


class MockToolBlock:
    """Mimics a tool_use content block."""
    def __init__(self, name, input_data, block_id="toolu_mock_001"):
        self.type = "tool_use"
        self.name = name
        self.input = input_data
        self.id = block_id


class MockTextBlock:
    """Mimics a text content block (returned when model declines to use a tool)."""
    def __init__(self, text):
        self.type = "text"
        self.text = text


class MockToolClient:
    """Simulates the Anthropic API for tool_use exercises.

    Tracks every call so we can inspect tool_choice, tools, and call_count.
    Supports preset responses for deterministic testing.

    Behavior by tool_choice mode:
      - auto:   returns text every 3rd call (simulates ~33% text fallback)
      - any:    always returns tool_use, picks first tool
      - forced: always returns the specified tool
    """
    def __init__(self):
        self.call_count = 0
        self.last_tool_choice = None
        self.last_tools = None
        self._preset_response = None
        self._preset_queue = []
        self.messages = type('Messages', (), {'create': self._create})()

    def set_response(self, data):
        """Set a single preset response for the next call."""
        self._preset_response = data

    def set_responses(self, data_list):
        """Set a queue of preset responses for sequential calls."""
        self._preset_queue = list(data_list)
        self.call_count = 0

    def _create(self, model="claude-sonnet-4-20250514", max_tokens=1024,
                tools=None, tool_choice=None, messages=None, **kwargs):
        self.call_count += 1
        self.last_tool_choice = tool_choice
        self.last_tools = tools

        tool_name = tools[0]["name"] if tools else "unknown"

        # Handle tool_choice behavior
        tc_type = tool_choice.get("type", "auto") if isinstance(tool_choice, dict) else tool_choice

        # Auto mode: sometimes return text instead of tool_use
        if tc_type == "auto" and self.call_count % 3 == 0:
            return MockToolResponse(
                [MockTextBlock("I couldn't extract structured data from this document. "
                               "The text appears to be too ambiguous.")],
                stop_reason="end_turn"
            )

        # Determine response data
        if self._preset_queue:
            idx = min(self.call_count - 1, len(self._preset_queue) - 1)
            data = self._preset_queue[idx]
        elif self._preset_response:
            data = self._preset_response
            self._preset_response = None
        else:
            # Default: realistic invoice extraction
            data = {
                "vendor_name": "Acme Corp",
                "invoice_number": "INV-2024-001",
                "total_amount": 1500.00,
                "currency": "USD",
                "currency_detail": None,
                "line_items": [
                    {"description": "Consulting services", "quantity": 10,
                     "unit_price": 100.00, "line_total": 1000.00},
                    {"description": "Materials", "quantity": 5,
                     "unit_price": 100.00, "line_total": 500.00}
                ],
                "confidence": "high"
            }

        # Forced tool_choice: use the specified tool name
        if tc_type == "tool" and isinstance(tool_choice, dict):
            tool_name = tool_choice.get("name", tool_name)

        return MockToolResponse([MockToolBlock(tool_name, data)])


client = MockToolClient()
print("Setup complete! Mock tool_use client ready.")
print("Replace MockToolClient() with anthropic.Anthropic() for production use.")

In [ ]:
#@title 🎧 Listen: Tool Schema
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_tool_schema.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 1: Defining a Tool Schema

The foundation of structured output is the **tool definition**. A tool has three parts:

1. **name** — A descriptive identifier (e.g., `extract_invoice_data`)
2. **description** — Tells Claude what this tool does and when to use it
3. **input_schema** — A JSON Schema that defines every field Claude must return

The schema is where the power lives. Every field you define — its type, its constraints, whether it is required — becomes a hard contract that Claude must satisfy.

Let us build a complete invoice extraction tool and walk through each design decision.

In [ ]:
# Complete invoice extraction tool with annotated schema
extraction_tool = {
    "name": "extract_invoice_data",
    "description": (
        "Extract structured data from an invoice document. "
        "Called once per invoice. All monetary amounts in decimal format."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            # --- Basic fields (required strings/numbers) ---
            "vendor_name": {
                "type": "string",
                "description": "Company or person name on the invoice"
            },
            "invoice_number": {
                "type": "string",
                "description": "Invoice or reference number exactly as printed"
            },
            "total_amount": {
                "type": "number",
                "description": "Total amount due (decimal, no currency symbols)"
            },

            # --- Enum field with "other" escape hatch ---
            "currency": {
                "type": "string",
                "enum": ["USD", "EUR", "GBP", "other"],
                "description": "Currency code. Use 'other' for uncommon currencies."
            },
            "currency_detail": {
                "type": ["string", "null"],
                "description": "If currency is 'other', the actual currency code (e.g., 'JPY'). Null otherwise."
            },

            # --- Nested array of objects ---
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "quantity": {"type": "integer"},
                        "unit_price": {"type": "number"},
                        "line_total": {"type": "number"}
                    },
                    "required": ["description", "quantity", "unit_price", "line_total"]
                },
                "description": "Individual line items on the invoice"
            },

            # --- Confidence field (model signals uncertainty) ---
            "confidence": {
                "type": "string",
                "enum": ["high", "medium", "low"],
                "description": "Extraction confidence. 'low' if any field is uncertain."
            }
        },
        "required": [
            "vendor_name", "invoice_number", "total_amount",
            "currency", "line_items", "confidence"
        ]
    }
}

# Pretty-print the schema
print("Invoice Extraction Tool Schema")
print("=" * 50)
print(f"Name: {extraction_tool['name']}")
print(f"Required fields: {extraction_tool['input_schema']['required']}")
print(f"Total properties: {len(extraction_tool['input_schema']['properties'])}")

print("\nField Details:")
print("-" * 50)
for field, spec in extraction_tool["input_schema"]["properties"].items():
    field_type = spec.get("type", spec.get("enum", "?"))
    required = field in extraction_tool["input_schema"]["required"]
    req_tag = " [REQUIRED]" if required else " [OPTIONAL]"

    if "enum" in spec:
        print(f"  {field:<20} enum{spec['enum']}{req_tag}")
    elif isinstance(field_type, list):
        print(f"  {field:<20} {' | '.join(field_type)}{req_tag}")
    else:
        print(f"  {field:<20} {field_type}{req_tag}")

print("\nDesign Decisions:")
print("  1. currency uses enum + 'other' -- handles uncommon currencies without forcing wrong match")
print("  2. currency_detail is nullable -- only populated when currency is 'other'")
print("  3. line_items is a nested array -- each item has its own required fields")
print("  4. confidence is an enum -- forces the model to pick exactly one of three levels")
print("  5. total_amount is 'number' not 'string' -- prevents '$1,500.00' format issues")

In [ ]:
#@title 🎧 Listen: Using Tool
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_using_tool.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Using the Tool for Extraction

Now let us call Claude with this tool and see what comes back. With `tool_choice` set to force `extract_invoice_data`, every response will be a schema-compliant tool call — no JSON parsing needed.

In [ ]:
# Call Claude with forced tool_use
sample_document = """
INVOICE
From: Acme Corp
Invoice #: INV-2024-001
Date: March 15, 2024

Item                    Qty    Unit Price    Total
Consulting services      10      $100.00    $1,000.00
Materials                 5      $100.00      $500.00
                                           ----------
                                   Total:  $1,500.00
Currency: USD
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[extraction_tool],
    tool_choice={"type": "tool", "name": "extract_invoice_data"},
    messages=[{
        "role": "user",
        "content": f"Extract the invoice data from this document:\n\n{sample_document}"
    }]
)

# The response is ALWAYS a tool_use block (never free text)
tool_block = response.content[0]
print(f"Response type: {tool_block.type}")
print(f"Tool called:   {tool_block.name}")
print(f"Stop reason:   {response.stop_reason}")
print()
print("Extracted data:")
print(json.dumps(tool_block.input, indent=2))

# Access fields directly -- no json.loads() needed!
data = tool_block.input
print(f"\nVendor: {data['vendor_name']}")
print(f"Total:  ${data['total_amount']:,.2f} {data['currency']}")
print(f"Items:  {len(data['line_items'])} line items")
print(f"Confidence: {data['confidence']}")

In [ ]:
#@title 🎧 Listen: Todo1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_todo1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

### TODO 1: Define a Job Posting Extraction Tool

Your turn. Define a tool schema for extracting structured data from a job posting.

**Required fields:**
- `company_name` (string)
- `job_title` (string)
- `location` (string)
- `employment_type` (enum: `"full_time"`, `"part_time"`, `"contract"`, `"other"`)
- `required_skills` (array of strings)
- `confidence` (enum: `"high"`, `"medium"`, `"low"`)

**Optional/nullable fields:**
- `salary_min` (nullable number — not always stated)
- `salary_max` (nullable number)
- `employment_detail` (string — for when type is `"other"`)
- `experience_years` (nullable integer — not always specified)

Think about:
- Which fields should be required vs optional?
- Why is `salary_min` nullable instead of just optional?
- What happens if the job posting mentions a currency you did not expect?

In [ ]:
# TODO 1: Define a job posting extraction tool
#
# Fill in the input_schema below. Requirements:
# - company_name, job_title, location: required strings
# - salary_min, salary_max: nullable numbers (use type: ["number", "null"])
# - employment_type: enum with "other" + employment_detail for specifics
# - required_skills: array of strings
# - experience_years: nullable integer
# - confidence: enum high/medium/low

job_tool = {
    "name": "extract_job_posting",
    "description": "Extract structured data from a job posting.",
    "input_schema": {
        "type": "object",
        "properties": {
            # YOUR SCHEMA HERE
            # Hint: follow the same patterns from the invoice tool
            # - Use "type": ["number", "null"] for nullable fields
            # - Use "enum": [..., "other"] for extensible categories
        },
        "required": []  # Which fields should be required?
    }
}

# Verify your schema
print(f"Tool name: {job_tool['name']}")
print(f"Properties defined: {len(job_tool['input_schema'].get('properties', {}))}")
print(f"Required fields: {job_tool['input_schema']['required']}")

In [ ]:
#@title 🎧 Listen: Todo1 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_todo1_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Todo2 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_todo2_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Todo3 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_todo3_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Todo4 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_17_todo4_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Todo5 Solution
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_19_todo5_solution.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

**Solution**

In [ ]:
# Solution for TODO 1
job_tool = {
    "name": "extract_job_posting",
    "description": (
        "Extract structured data from a job posting or listing. "
        "Use null for salary and experience if not explicitly stated in the posting."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "company_name": {
                "type": "string",
                "description": "Name of the hiring company"
            },
            "job_title": {
                "type": "string",
                "description": "Job title exactly as listed"
            },
            "location": {
                "type": "string",
                "description": "Job location (city, state, or 'Remote')"
            },
            "salary_min": {
                "type": ["number", "null"],
                "description": "Minimum salary in annual USD. Null if not stated."
            },
            "salary_max": {
                "type": ["number", "null"],
                "description": "Maximum salary in annual USD. Null if not stated."
            },
            "employment_type": {
                "type": "string",
                "enum": ["full_time", "part_time", "contract", "other"],
                "description": "Employment type. Use 'other' for internship, freelance, etc."
            },
            "employment_detail": {
                "type": ["string", "null"],
                "description": "If employment_type is 'other', specify here (e.g., 'internship'). Null otherwise."
            },
            "required_skills": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of required skills/technologies mentioned"
            },
            "experience_years": {
                "type": ["integer", "null"],
                "description": "Minimum years of experience required. Null if not stated."
            },
            "confidence": {
                "type": "string",
                "enum": ["high", "medium", "low"],
                "description": "Extraction confidence. 'low' if job posting is ambiguous."
            }
        },
        "required": [
            "company_name", "job_title", "location",
            "employment_type", "required_skills", "confidence"
        ]
    }
}

# Verify the schema
print("Job Posting Extraction Tool")
print("=" * 50)
print(f"Required fields ({len(job_tool['input_schema']['required'])}):")
for f in job_tool["input_schema"]["required"]:
    print(f"  - {f}")

print(f"\nNullable fields:")
for field, spec in job_tool["input_schema"]["properties"].items():
    if isinstance(spec.get("type"), list) and "null" in spec["type"]:
        print(f"  - {field}: {spec['type']}")

print(f"\nEnum fields:")
for field, spec in job_tool["input_schema"]["properties"].items():
    if "enum" in spec:
        print(f"  - {field}: {spec['enum']}")

print("\nKey design decisions:")
print("  1. salary_min/max are NULLABLE, not optional -- forces explicit null vs omission")
print("  2. employment_type has 'other' + employment_detail for internships, freelance, etc.")
print("  3. experience_years is nullable integer -- '3+ years' becomes 3, unstated becomes null")
print("  4. required_skills is an array -- handles any number of skills cleanly")

# Test it
client.set_response({
    "company_name": "TechCorp",
    "job_title": "Senior ML Engineer",
    "location": "San Francisco, CA",
    "salary_min": 180000,
    "salary_max": 250000,
    "employment_type": "full_time",
    "employment_detail": None,
    "required_skills": ["Python", "PyTorch", "Distributed Systems", "Kubernetes"],
    "experience_years": 5,
    "confidence": "high"
})

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[job_tool],
    tool_choice={"type": "tool", "name": "extract_job_posting"},
    messages=[{"role": "user", "content": "Extract: Senior ML Engineer at TechCorp, SF, $180-250k, 5+ yrs, Python/PyTorch/K8s"}]
)

result = response.content[0].input
print("\nTest extraction:")
print(json.dumps(result, indent=2))

In [ ]:
#@title 🎧 Listen: Tool Choice Modes
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_tool_choice_modes.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 2: tool_choice Deep Dive

The `tool_choice` parameter is one of the most frequently tested concepts on the exam. It controls whether Claude *must* use a tool, *may* use a tool, or *must* use a *specific* tool.

### The Three Modes

**1. `auto` (default)** — Claude decides whether to use a tool.
```python
tool_choice={"type": "auto"}
```
Good for chatbots where the user might ask a question (no tool needed) or request an extraction (tool needed). The risk: Claude might return free text when you expected structured data.

**2. `any`** — Claude *must* call a tool, but picks which one.
```python
tool_choice={"type": "any"}
```
Good when you have multiple extraction schemas and want Claude to route to the right one. For example: invoice tool, receipt tool, contract tool — let Claude pick based on the document type.

**3. `forced` (specific tool)** — Claude *must* call this exact tool.
```python
tool_choice={"type": "tool", "name": "extract_invoice_data"}
```
Good for extraction pipelines where you already know the document type. Eliminates all ambiguity — the response is *always* a tool call with the right schema.

In [ ]:
#@title 🎧 Listen: Todo2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_todo2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### TODO 2: Test All Three tool_choice Modes

Run the same extraction request with each tool_choice mode and observe the differences. Pay attention to:
- Does the response contain a tool call or free text?
- Which tool was called?
- How reliable is each mode for extraction pipelines?

In [ ]:
# TODO 2: Test all three tool_choice modes on the same input
#
# For each mode:
#   1. Reset client.call_count = 0
#   2. Make 6 calls with the same input
#   3. Count how many returned tool_use vs text
#   4. Record the results

document = "Invoice from Acme Corp, INV-2024-001, Total $1,500 USD"

modes = {
    "auto": {"type": "auto"},
    "any": {"type": "any"},
    "forced": {"type": "tool", "name": "extract_invoice_data"}
}

# YOUR CODE HERE:
# For each mode, make 6 calls and count tool_use vs text responses
# Print a comparison table at the end

# Hint: Check response.content[0].type to see if it's "tool_use" or "text"
# Hint: Check response.stop_reason -- "tool_use" vs "end_turn"

results = {}
for mode_name, mode_config in modes.items():
    pass  # Replace with your implementation

**Solution**

In [ ]:
# Solution for TODO 2
document = "Invoice from Acme Corp, INV-2024-001, Total $1,500 USD"

modes = {
    "auto": {"type": "auto"},
    "any": {"type": "any"},
    "forced": {"type": "tool", "name": "extract_invoice_data"}
}

results = {}

for mode_name, mode_config in modes.items():
    client.call_count = 0  # Reset counter
    tool_use_count = 0
    text_count = 0

    for i in range(6):
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1024,
            tools=[extraction_tool],
            tool_choice=mode_config,
            messages=[{"role": "user", "content": f"Extract data:\n{document}"}]
        )

        block = response.content[0]
        if block.type == "tool_use":
            tool_use_count += 1
        else:
            text_count += 1

    results[mode_name] = {
        "tool_use": tool_use_count,
        "text": text_count,
        "reliability": tool_use_count / 6
    }

# Print comparison table
print("tool_choice Mode Comparison (6 calls each)")
print("=" * 65)
print(f"{'Mode':<12} {'tool_use':<12} {'text':<12} {'Reliability':<15} {'Use Case'}")
print("-" * 65)

use_cases = {
    "auto": "Chatbots with optional tools",
    "any": "Multi-schema routing",
    "forced": "Extraction pipelines"
}

for mode_name, r in results.items():
    print(f"{mode_name:<12} {r['tool_use']:<12} {r['text']:<12} "
          f"{r['reliability']:.0%}{'':>10} {use_cases[mode_name]}")

print()
print("Key observations:")
print(f"  1. 'auto' returned text {results['auto']['text']}/6 times -- unreliable for extraction")
print(f"  2. 'any' returned tool_use {results['any']['tool_use']}/6 times -- reliable but model picks the tool")
print(f"  3. 'forced' returned tool_use {results['forced']['tool_use']}/6 times -- guaranteed structured output")
print()
print("Rule of thumb:")
print("  - Building an extraction pipeline? Use FORCED.")
print("  - Building a chatbot with multiple tools? Use ANY or AUTO.")
print("  - Need the model to decide if extraction is possible? Use AUTO.")

In [ ]:
#@title 🎧 Listen: Schema Patterns
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_schema_patterns.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 3: Schema Design Patterns

Defining a schema is easy. Defining a *good* schema requires knowing three patterns that prevent common failures in production.

### Pattern 1: Nullable Fields (Prevent Fabrication)

When data might not exist in the source document, make the field **nullable** instead of optional:

```python
# BAD: Optional field -- model might skip it OR fabricate a value
"salary": {"type": "number"}

# GOOD: Nullable field -- model must explicitly return null if unknown
"salary": {"type": ["number", "null"]}
```

Why nullable over optional? Because `null` is an explicit signal: "I looked for this data and it is not there." An omitted field is ambiguous — did the model forget, or is the data missing?

### Pattern 2: Enum + "other" (Handle Unexpected Categories)

Real-world data does not fit neat categories. The enum + "other" pattern handles this gracefully:

```python
"currency": {
    "enum": ["USD", "EUR", "GBP", "other"]
},
"currency_detail": {
    "type": ["string", "null"],
    "description": "If currency is 'other', specify here. Null otherwise."
}
```

Without the "other" escape hatch, Claude must force-match to the closest enum value — turning JPY into USD, for example. With "other", it can honestly say "this does not fit your categories" and provide the actual value.

### Pattern 3: Confidence Fields (Signal Uncertainty)

A confidence field lets the model tell you when it is unsure:

```python
"confidence": {
    "type": "string",
    "enum": ["high", "medium", "low"],
    "description": "Extraction confidence. 'low' if any field is uncertain or source is ambiguous."
}
```

This is crucial for human-in-the-loop systems. Route "high" confidence extractions to automated processing; route "low" confidence to human review. This single field can reduce error rates by 50%+ in production.

In [ ]:
# Demonstrate all three patterns with concrete examples

print("=" * 60)
print("PATTERN 1: Nullable Fields -- Prevent Fabrication")
print("=" * 60)

# Scenario: extracting from a receipt that has no vendor name
client.set_response({
    "vendor_name": None,        # Nullable: explicitly signals "not found"
    "invoice_number": "REC-001",
    "total_amount": 42.50,
    "currency": "USD",
    "currency_detail": None,
    "line_items": [
        {"description": "Coffee", "quantity": 1, "unit_price": 42.50, "line_total": 42.50}
    ],
    "confidence": "medium"      # Medium because vendor is missing
})

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[extraction_tool],
    tool_choice={"type": "tool", "name": "extract_invoice_data"},
    messages=[{"role": "user", "content": "Extract from: Receipt #REC-001, Coffee $42.50, no vendor name visible"}]
)

data = response.content[0].input
print(f"  vendor_name: {data['vendor_name']}  (null = not found, NOT fabricated)")
print(f"  confidence:  {data['confidence']}  (medium because vendor is missing)")
print(f"  total:       ${data['total_amount']}")
print()

print("=" * 60)
print("PATTERN 2: Enum + 'other' -- Handle Unexpected Categories")
print("=" * 60)

# Scenario: invoice in Japanese Yen (not in our enum)
client.set_response({
    "vendor_name": "Tokyo Trading Co",
    "invoice_number": "INV-JP-2024",
    "total_amount": 150000,
    "currency": "other",         # Enum value: signals non-standard currency
    "currency_detail": "JPY",    # Detail field: provides the actual currency
    "line_items": [
        {"description": "Components", "quantity": 100, "unit_price": 1500, "line_total": 150000}
    ],
    "confidence": "high"
})

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    tools=[extraction_tool],
    tool_choice={"type": "tool", "name": "extract_invoice_data"},
    messages=[{"role": "user", "content": "Extract: Invoice INV-JP-2024 from Tokyo Trading, 150,000 JPY"}]
)

data = response.content[0].input
print(f"  currency:        {data['currency']}  (enum value)")
print(f"  currency_detail: {data['currency_detail']}  (actual currency when 'other')")
print(f"  Without 'other', the model would force-match JPY to USD -- silently wrong!")
print()

print("=" * 60)
print("PATTERN 3: Confidence Fields -- Signal Uncertainty")
print("=" * 60)

confidence_examples = [
    ("high", "Clear invoice with all fields visible"),
    ("medium", "Blurry scan with partially readable total"),
    ("low", "Handwritten receipt, multiple possible totals"),
]

for level, scenario in confidence_examples:
    print(f"  [{level.upper():<6}] {scenario}")

print("\n  Routing strategy:")
print("    high   -> Automated processing (no human review)")
print("    medium -> Automated processing + flagged for spot-check")
print("    low    -> Routed to human reviewer before processing")

In [ ]:
#@title 🎧 Listen: Todo3
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_todo3.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### TODO 3: Design a Medical Record Extraction Schema

Design a tool schema for extracting data from medical records. Your schema must use **all three patterns**:

1. **Nullable fields** -- blood_type (not always in records)
2. **Enum + "other"** -- diagnosis_code (common codes + fallback)
3. **Confidence field** -- extraction_confidence

**Required fields:**
- patient_name (string)
- date_of_birth (string, YYYY-MM-DD format)
- diagnosis_code (enum: E11 diabetes, I10 hypertension, J45 asthma, M54 back pain, other)
- diagnosis_detail (nullable string -- for when code is "other")
- blood_type (nullable string -- not always recorded)
- medications (array of objects: name, dosage, frequency)
- extraction_confidence (enum: high/medium/low)

In [ ]:
# TODO 3: Define a medical record extraction tool
#
# Use ALL THREE patterns:
#   - Nullable fields for blood_type and diagnosis_detail
#   - Enum + "other" for diagnosis_code
#   - Confidence field for extraction_confidence

medical_tool = {
    "name": "extract_medical_record",
    "description": "Extract structured data from a medical record.",
    "input_schema": {
        "type": "object",
        "properties": {
            # YOUR SCHEMA HERE
            # Pattern 1 (nullable): blood_type can be ["string", "null"]
            # Pattern 2 (enum + other): diagnosis_code with "other" + diagnosis_detail
            # Pattern 3 (confidence): extraction_confidence enum
        },
        "required": []  # Which fields should be required?
    }
}

# Verify
print(f"Tool: {medical_tool['name']}")
print(f"Properties: {len(medical_tool['input_schema'].get('properties', {}))}")
print(f"Required: {medical_tool['input_schema']['required']}")

**Solution**

In [ ]:
# Solution for TODO 3
medical_tool = {
    "name": "extract_medical_record",
    "description": (
        "Extract structured patient data from a medical record. "
        "Use null for fields not present in the record. "
        "Use 'other' for diagnosis codes not in the common set."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "patient_name": {
                "type": "string",
                "description": "Full patient name as written in the record"
            },
            "date_of_birth": {
                "type": "string",
                "description": "Date of birth in YYYY-MM-DD format"
            },
            "diagnosis_code": {
                "type": "string",
                "enum": ["E11", "I10", "J45", "M54", "other"],
                "description": "ICD-10 code. E11=diabetes, I10=hypertension, J45=asthma, M54=back pain."
            },
            "diagnosis_detail": {
                "type": ["string", "null"],
                "description": "If diagnosis_code is 'other', the actual diagnosis. Null otherwise."
            },
            "blood_type": {
                "type": ["string", "null"],
                "description": "Patient blood type (e.g., 'A+', 'O-'). Null if not in record."
            },
            "medications": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {"type": "string", "description": "Medication name"},
                        "dosage": {"type": "string", "description": "Dosage (e.g., '500mg')"},
                        "frequency": {"type": "string", "description": "How often (e.g., 'twice daily')"}
                    },
                    "required": ["name", "dosage", "frequency"]
                },
                "description": "Current medications list"
            },
            "extraction_confidence": {
                "type": "string",
                "enum": ["high", "medium", "low"],
                "description": "Confidence. 'low' if record is handwritten or partially illegible."
            }
        },
        "required": [
            "patient_name", "date_of_birth", "diagnosis_code",
            "medications", "extraction_confidence"
        ]
    }
}

# Verify all three patterns
print("Medical Record Extraction Tool")
print("=" * 50)
nullable = [f for f, s in medical_tool["input_schema"]["properties"].items()
            if isinstance(s.get("type"), list) and "null" in s["type"]]
print(f"Pattern 1 -- Nullable fields: {nullable}")

enums = {f: s["enum"] for f, s in medical_tool["input_schema"]["properties"].items() if "enum" in s}
has_other = {f: "other" in e for f, e in enums.items()}
print(f"Pattern 2 -- Enum fields: {enums}")
print(f"           Has 'other': {has_other}")

conf = [f for f in enums if "confidence" in f.lower()]
print(f"Pattern 3 -- Confidence field: {conf}")
print(f"All three patterns: {len(nullable) > 0 and any(has_other.values()) and len(conf) > 0}")

# Test
client.set_response({
    "patient_name": "Jane Smith", "date_of_birth": "1985-07-22",
    "diagnosis_code": "E11", "diagnosis_detail": None, "blood_type": None,
    "medications": [
        {"name": "Metformin", "dosage": "500mg", "frequency": "twice daily"},
        {"name": "Lisinopril", "dosage": "10mg", "frequency": "once daily"}
    ],
    "extraction_confidence": "medium"
})
response = client.messages.create(
    model="claude-sonnet-4-20250514", max_tokens=1024,
    tools=[medical_tool],
    tool_choice={"type": "tool", "name": "extract_medical_record"},
    messages=[{"role": "user", "content": "Extract patient data"}]
)
print("\nTest extraction:")
print(json.dumps(response.content[0].input, indent=2))
print("\nblood_type is null (not fabricated), confidence is 'medium' (signaling uncertainty)")

In [ ]:
#@title 🎧 Listen: Syntax Vs Semantic Deep
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_15_syntax_vs_semantic_deep.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 4: Syntax Errors vs Semantic Errors

This is the single most important distinction in this notebook, and it is directly tested on the exam.

**Syntax errors** are structural problems -- malformed JSON, wrong field names, wrong types. Schemas **eliminate** these entirely.

**Semantic errors** are meaning problems -- values that are structurally valid but factually wrong. Schemas **cannot catch** these.

### Example: A Perfectly Structured, Completely Wrong Extraction

```json
{
    "vendor_name": "Acme Corp",
    "total_amount": 500.00,
    "line_items": [
        {"description": "Consulting", "quantity": 10, "unit_price": 100.00, "line_total": 1000.00},
        {"description": "Materials", "quantity": 5, "unit_price": 100.00, "line_total": 500.00}
    ]
}
```

This passes schema validation perfectly. But there are two semantic errors:
1. total_amount is 500.00, but the line items sum to 1500.00
2. The line items each compute correctly (10 * 100 = 1000, 5 * 100 = 500), but the total is wrong

You need **validation logic** -- code that checks mathematical relationships, business rules, and logical consistency -- to catch these. That is the subject of Notebook 3.

In [ ]:
#@title 🎧 Listen: Todo4
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_16_todo4.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### TODO 4: Build a Semantic Error Checker

Write a function that takes an extraction result (from the invoice tool) and checks for semantic errors that the schema cannot catch:

1. **Math check**: Each line item's quantity * unit_price should equal line_total (within $0.01)
2. **Sum check**: All line_total values should sum to total_amount (within $0.01)
3. **Confidence check**: If any key field is null, confidence should not be "high"
4. **Currency check**: If currency is "other", currency_detail should not be null

In [ ]:
# TODO 4: Write a semantic error checker
#
# Input: dict from extract_invoice_data tool
# Output: list of error strings (empty = no errors)

def check_semantic_errors(extraction):
    """Check for semantic errors that schemas cannot catch.

    Returns a list of human-readable error strings.
    Empty list means no semantic errors detected.
    """
    errors = []

    # Check 1: Each line item's quantity * unit_price should equal line_total
    # YOUR CODE HERE

    # Check 2: Sum of line_totals should equal total_amount
    # YOUR CODE HERE

    # Check 3: If any key field is None, confidence should not be "high"
    # YOUR CODE HERE

    # Check 4: If currency is "other", currency_detail should not be None
    # YOUR CODE HERE

    return errors


# Quick test
print("Function defined. Run the solution cell to see it in action.")

**Solution**

In [ ]:
# Solution for TODO 4

def check_semantic_errors(extraction):
    """Check for semantic errors that schemas cannot catch."""
    errors = []

    # Check 1: Line item math
    for i, item in enumerate(extraction.get("line_items", [])):
        expected = item["quantity"] * item["unit_price"]
        actual = item["line_total"]
        if abs(expected - actual) > 0.01:
            errors.append(
                f"Line item {i+1} ('{item['description']}'): "
                f"quantity({item['quantity']}) * unit_price(${item['unit_price']:.2f}) = "
                f"${expected:.2f}, but line_total is ${actual:.2f}"
            )

    # Check 2: Sum of line_totals should equal total_amount
    line_sum = sum(item["line_total"] for item in extraction.get("line_items", []))
    total = extraction.get("total_amount", 0)
    if abs(line_sum - total) > 0.01:
        errors.append(f"Line items sum to ${line_sum:.2f} but total_amount is ${total:.2f}")

    # Check 3: Null fields with high confidence
    null_fields = [k for k, v in extraction.items()
                   if v is None and k not in ("currency_detail",)]
    if null_fields and extraction.get("confidence") == "high":
        errors.append(f"Confidence is 'high' but these fields are null: {null_fields}")

    # Check 4: "other" currency needs detail
    if extraction.get("currency") == "other" and extraction.get("currency_detail") is None:
        errors.append("Currency is 'other' but currency_detail is null.")

    return errors


# Test 1: Correct extraction
print("Test 1: Correct extraction")
print("-" * 40)
correct = {
    "vendor_name": "Acme Corp", "invoice_number": "INV-2024-001",
    "total_amount": 1500.00, "currency": "USD", "currency_detail": None,
    "line_items": [
        {"description": "Consulting", "quantity": 10, "unit_price": 100.00, "line_total": 1000.00},
        {"description": "Materials", "quantity": 5, "unit_price": 100.00, "line_total": 500.00}
    ],
    "confidence": "high"
}
errors = check_semantic_errors(correct)
print(f"  Errors: {len(errors)}")
assert len(errors) == 0
print("  PASSED\n")

# Test 2: Semantic errors
print("Test 2: Multiple semantic errors")
print("-" * 40)
wrong = {
    "vendor_name": "Acme", "invoice_number": "INV-001",
    "total_amount": 500.00, "currency": "other", "currency_detail": None,
    "line_items": [
        {"description": "Consulting", "quantity": 10, "unit_price": 100.00, "line_total": 800.00},
        {"description": "Materials", "quantity": 5, "unit_price": 100.00, "line_total": 500.00}
    ],
    "confidence": "high"
}
errors = check_semantic_errors(wrong)
print(f"  Errors found: {len(errors)}")
for e in errors:
    print(f"    - {e}")
assert len(errors) >= 3
print("  PASSED\n")

# Test 3: Null with high confidence
print("Test 3: Null field with high confidence")
print("-" * 40)
suspicious = {
    "vendor_name": None, "invoice_number": "INV-001",
    "total_amount": 100.00, "currency": "USD", "currency_detail": None,
    "line_items": [{"description": "Svc", "quantity": 1, "unit_price": 100.00, "line_total": 100.00}],
    "confidence": "high"
}
errors = check_semantic_errors(suspicious)
print(f"  Errors found: {len(errors)}")
for e in errors:
    print(f"    - {e}")
assert len(errors) >= 1
print("  PASSED\n")

print("All semantic error checks passed!")
print("\nKey insight: Every one of these extractions passes schema validation.")
print("The JSON is perfectly structured. The VALUES are wrong.")
print("That is the gap that semantic validation fills.")

In [ ]:
#@title 🎧 Listen: Todo5
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_18_todo5.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Section 5: Putting It All Together

Now let us combine everything into a complete extraction function that:
1. Uses forced tool_choice for guaranteed structured output
2. Extracts data via tool_use
3. Checks for semantic errors
4. Returns a comprehensive result object with metadata

### TODO 5 (Integration): Build a Complete Structured Extraction Function

Build structured_extract() that combines tool_use with semantic validation. The function should:
1. Call the API with forced tool_choice
2. Extract the tool_use result
3. Run semantic validation
4. Return a result dict with data, schema_valid, semantic_valid, semantic_errors, and needs_validation_retry

In [ ]:
# TODO 5: Build a complete structured extraction function

def structured_extract(document, tool_schema, client):
    """Extract structured data with forced tool_use and semantic validation."""
    # Step 1: Call API with forced tool_choice
    # YOUR CODE HERE

    # Step 2: Extract the tool_use result
    # YOUR CODE HERE

    # Step 3: Run semantic validation
    # YOUR CODE HERE

    # Step 4: Return comprehensive result
    # YOUR CODE HERE
    pass


print("Function defined. Run the solution cell to see it in action.")

**Solution**

In [ ]:
# Solution for TODO 5

def structured_extract(document, tool_schema, client):
    """Extract structured data with forced tool_use and semantic validation."""
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1024,
        tools=[tool_schema],
        tool_choice={"type": "tool", "name": tool_schema["name"]},
        messages=[{"role": "user", "content": f"Extract data:\n\n{document}"}]
    )
    tool_block = response.content[0]
    extracted = tool_block.input
    semantic_errors = check_semantic_errors(extracted)
    return {
        "data": extracted,
        "schema_valid": True,
        "semantic_valid": len(semantic_errors) == 0,
        "semantic_errors": semantic_errors,
        "needs_validation_retry": len(semantic_errors) > 0
    }


# Test 1: Clean extraction
print("Test 1: Clean extraction (no errors)")
print("=" * 50)
client.set_response({
    "vendor_name": "TechSupply Inc", "invoice_number": "INV-2024-042",
    "total_amount": 750.00, "currency": "USD", "currency_detail": None,
    "line_items": [
        {"description": "Server rack", "quantity": 1, "unit_price": 500.00, "line_total": 500.00},
        {"description": "Cables", "quantity": 5, "unit_price": 50.00, "line_total": 250.00}
    ],
    "confidence": "high"
})
result = structured_extract("Invoice from TechSupply", extraction_tool, client)
print(f"  Schema valid:    {result['schema_valid']}")
print(f"  Semantic valid:  {result['semantic_valid']}")
print(f"  Errors:          {result['semantic_errors']}")
print(f"  Needs retry:     {result['needs_validation_retry']}")
print()

# Test 2: Semantic errors
print("Test 2: Extraction with semantic errors")
print("=" * 50)
client.set_response({
    "vendor_name": "BadMath Corp", "invoice_number": "INV-099",
    "total_amount": 200.00, "currency": "other", "currency_detail": None,
    "line_items": [
        {"description": "Widget A", "quantity": 3, "unit_price": 50.00, "line_total": 150.00},
        {"description": "Widget B", "quantity": 2, "unit_price": 100.00, "line_total": 200.00}
    ],
    "confidence": "high"
})
result = structured_extract("Invoice from BadMath", extraction_tool, client)
print(f"  Schema valid:    {result['schema_valid']}  (always True with forced tool_use)")
print(f"  Semantic valid:  {result['semantic_valid']}")
print(f"  Needs retry:     {result['needs_validation_retry']}")
print(f"  Errors:")
for e in result['semantic_errors']:
    print(f"    - {e}")
print()
print("Key insight: Schema says VALID. Semantic validation says INVALID.")
print("Both are needed. Schemas catch structure. Validation catches meaning.")
print("\nIn Notebook 3, we build the retry loop that feeds errors back to Claude.")

---

## Visualization: Schema Compliance and Error Types

Let us visualize two key insights:
1. How tool_choice mode affects schema compliance rates
2. The breakdown of error types when using forced tool_choice

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left chart: Schema compliance by tool_choice mode
modes = ["auto", "any", "forced"]
structured_rates = [0.92, 1.00, 1.00]
text_fallback    = [0.08, 0.00, 0.00]

x = range(len(modes))
bars1 = ax1.bar(x, structured_rates, color='#2196F3', edgecolor='black',
                alpha=0.85, label='Schema-compliant (tool_use)')
bars2 = ax1.bar(x, text_fallback, bottom=structured_rates, color='#f44336',
                edgecolor='black', alpha=0.85, label='Free text (no schema)')
ax1.set_xticks(x)
ax1.set_xticklabels(['auto', 'any', 'forced'], fontsize=10)
ax1.set_ylabel('Response Rate', fontsize=12)
ax1.set_title('Schema Compliance by tool_choice Mode', fontsize=13)
ax1.set_ylim(0, 1.15)
ax1.legend(fontsize=10, loc='upper left')
ax1.grid(True, alpha=0.3, axis='y')

for bar, rate in zip(bars1, structured_rates):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
             f'{rate:.0%}', ha='center', va='center', fontsize=13,
             fontweight='bold', color='white')
for bar, rate, bottom in zip(bars2, text_fallback, structured_rates):
    if rate > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, bottom + rate/2,
                 f'{rate:.0%}', ha='center', va='center', fontsize=11,
                 fontweight='bold', color='white')

# Right chart: Error types
categories = ['Syntax Errors', 'Semantic Errors']
with_schema = [0, 8]
without_schema = [12, 8]
x2 = range(len(categories))
width = 0.35
bars3 = ax2.bar([i - width/2 for i in x2], without_schema, width,
                color='#f44336', edgecolor='black', alpha=0.85, label='Without tool_use')
bars4 = ax2.bar([i + width/2 for i in x2], with_schema, width,
                color='#4CAF50', edgecolor='black', alpha=0.85, label='With forced tool_use')
ax2.set_xticks(x2)
ax2.set_xticklabels(categories, fontsize=11)
ax2.set_ylabel('Error Rate (%)', fontsize=12)
ax2.set_title('Error Types: With vs Without tool_use', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim(0, 18)

for bar, val in zip(bars3, without_schema):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val}%', ha='center', fontsize=11, fontweight='bold')
for bar, val in zip(bars4, with_schema):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("Left: 'auto' has 8% chance of returning free text. 'any'/'forced' give 100%.")
print("Right: tool_use eliminates ALL syntax errors (12% -> 0%).")
print("       Semantic errors remain at ~8% -- they require validation logic.")

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_20_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

---

## Key Takeaways

1. **tool_use forces schema-compliant output.** Define a tool with a JSON schema and Claude must return data matching that schema. No more parsing free-text JSON.

2. **tool_choice controls when tools are used:**
   - "auto" -- model decides (good for chatbots with optional tools)
   - "any" -- must call a tool, model picks which one (good for multi-schema routing)
   - forced -- must call this specific tool (good for extraction pipelines)

3. **Schemas eliminate syntax errors, not semantic errors.** The JSON will always be valid. Field types will always match. But total_amount: 500 when it should be 450 is a semantic error that requires validation.

4. **Three schema design patterns prevent common failures:**
   - **Nullable fields** prevent fabrication when data is missing
   - **Enum + "other"** handles unexpected categories without forcing wrong matches
   - **Confidence fields** let the model signal uncertainty for human review

5. **Always use forced tool_choice for extraction pipelines.** The 8% failure rate with "auto" becomes 0% with forced. For extraction, there is no reason to accept free-text responses.

---

## What's Next

In **Notebook 3: Validation-Retry Loops**, we tackle the semantic errors that schemas cannot catch. We will build extract-validate-retry pipelines that automatically detect and correct errors like math mismatches and impossible dates.

---

*Built with Vizuara -- AI education that meets you where you are.*